In [44]:
import pandas as pd
import numpy as np

In [45]:
read = pd.read_csv("./soil_data.csv")
read = read.transpose()
read = read.drop(read.index[-1])

In [46]:
data = read.reset_index()
data.columns = data.iloc[0]
data = data.iloc[1:]

# dropping 2026-04-27/15:34:10
# data = data.drop("Date", axis=1)
# data[data["Date"].str.startswith("2026-04-27") & data["Time"].str.startswith("15:34:10")]

# meta_col = data.columns[data.columns.notna()]

In [47]:
count = 0
new = []
for i in data.columns:
    if(pd.isna(i)):
        new.append("spec"+str(count))
        count+=1
    else:
        new.append(i)
data.columns = new
data = data.drop(5)
data = data[~(data["Date"].str.startswith("2026-04-27") & data["Time"].str.startswith("15:34:10"))]
data=data.reset_index(drop=True)

In [48]:
selected = data.drop(columns=['Date', 'Time', 'Reading', 'Reading Label', 'Exposure Number',
       'Total Exposure Number', 'Energy Slope', 'Energy Offset', 'Real Time',
       'Hardware LiveTime', 'Live Time', 'Probe Serial Number', 'First Rate',
       'Last Rate', 'Total Counts', 'Total Rate', 'Ambient Pressure',
       'Probe Temperature (°C)', 'Tube Current Set', 'Tube Current Monitor',
       'Tube Voltage Set', 'Tube Voltage Monitor', 'Lt Multiplier',
       'Col Multiplier', 'Step Index', 'Step Total', 'Filter Material',
       'Filter Diameter', 'Filter Thickness', 'Reset Time', 'Fast Counts',
       'Top Resets', 'Reset Count', 'Width Rejects', 'Pileups', 'Clip Top',
       'Clip Bottom', 'Input Count Rate', 'Output Count Rate', 'Data'])

In [49]:
reshaped = selected.to_numpy().reshape(selected.shape[0]//2, selected.shape[1]*2)
x = pd.DataFrame(reshaped, columns=["s"+str(i) for i in range(2047)]+["S"+str(i) for i in range(2047)])
x = x.astype(int)
x
# x = x.loc[:, (x != 0).any()]

,s0,s1,s2,s3,s4,s5,s6,s7,s8,s9,...,S2037,S2038,S2039,S2040,S2041,S2042,S2043,S2044,S2045,S2046
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
76,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
77,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
78,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [50]:
x_heavy = x[[c for c in x.columns if c[0]=='s']]
x_heavy = x_heavy[x_heavy.columns[:800]]

In [51]:
# results
read_result = pd.read_csv("soil_result.csv")
read_result = read_result.drop(2)
read_result = read_result[~(read_result["Date"].str.startswith("2026-04-27") & read_result["Time"].str.startswith("15:34:10"))]
read_result = read_result.reset_index(drop=True)

In [52]:
result = read_result[read_result.columns[11:-8]]
result = result[[c for c in result.columns if c.split()[-1]=="Concentration"]]
result = result.replace("<LOD", "0").astype(int)
result

,Mg Concentration,Al Concentration,Si Concentration,P Concentration,S Concentration,Cl Concentration,Ca Concentration,Ti Concentration,V Concentration,Cr Concentration,...,Sb Concentration,Ba Concentration,W Concentration,Au Concentration,Hg Concentration,Pb Concentration,Bi Concentration,Th Concentration,U Concentration,LE Concentration
0,0,6917,43170,165,702,1721,5818,3840,110,0,...,0,0,41,0,0,46,0,0,0,914257
1,0,7007,54659,0,960,2253,6314,5280,0,0,...,0,0,28,0,0,58,0,18,0,895380
2,0,0,32071,0,582,2208,6377,4840,127,84,...,0,0,39,0,0,47,0,0,0,920121
3,0,7874,45256,532,808,2835,8078,3785,0,0,...,0,0,38,0,0,91,0,17,0,901849
4,0,12550,49416,245,802,2562,3029,4460,0,80,...,0,0,28,0,0,124,0,0,0,897794
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,0,0,6843,3755,3124,7578,16216,0,0,0,...,0,0,19,0,0,5,0,68,16,960583
76,0,2214,5137,1296,1116,5362,11921,0,0,0,...,0,0,19,0,0,3,0,50,10,972545
77,0,1644,4259,2261,2114,11013,11820,0,0,0,...,0,0,20,0,0,4,0,55,10,966381
78,0,2464,3945,2035,1607,10222,11147,0,0,0,...,0,0,13,0,0,0,0,54,10,968110


In [53]:
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import mean_absolute_percentage_error
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
# from sklearn.metrics import mean_absolute_error
# from sklearn.metrics import mean_absolute_percentage_error
# from sklearn.metrics import mean_squared_log_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import KFold, cross_validate

In [54]:
# pca = PCA(n_components=20)
# scaler = StandardScaler()
rf = RandomForestRegressor(n_estimators=200, random_state=130)
svr = SVR(kernel="linear", C=50)
lr = LinearRegression()
pls = PLSRegression(n_components=20, scale=False)

In [ ]:
X = x_heavy
# X = scaler.fit_transform(X)
# X = pca.fit_transform(X)
Y = result[["Cu Concentration", "Zn Concentration", "As Concentration", "Pb Concentration"]]
cv = KFold(n_splits=5, shuffle=True, random_state=42)
# train_X, val_X, train_y, val_y = train_test_split(x_heavy, result[["Cu Concentration", "Zn Concentration", "As Concentration", "Pb Concentration"]])
models = [rf, MultiOutputRegressor(svr), lr, pls]
# try: Lasso
for m in models:
    score = cross_validate(m, X, Y, cv=cv, scoring={"mae":"neg_mean_absolute_error", "mse":"neg_mean_squared_error", "r2":"r2"})
    print("mae: ", -score["test_mae"].mean())
    print("mse: ", -score["test_mse"].mean())
    print("r2: ", score["test_r2"].mean())
    print()

mae:  39.957984374999995
mse:  8685.067773046872
r2:  0.6405360889106004

mae:  21.44253288306968
mse:  1466.2473052393923
r2:  0.8467874212708875

mae:  21.44364532299279
mse:  1466.61394063582
r2:  0.8480525833172118

mae:  21.62207511208046
mse:  1454.9238490201603
r2:  0.8009141436305056



In [56]:
# individual element
ele = ["Cu Concentration", "Zn Concentration", "As Concentration", "Pb Concentration"]
models = [rf, svr, lr, pls]
for m in models:
    for e in ele:
        score = cross_validate(m, X, result[e], cv=cv, scoring={"mae":"neg_mean_absolute_error", "mse":"neg_mean_squared_error", "r2":"r2"})
        print(e)
        print("mae: ", -score["test_mae"].mean())
        print("mse: ", -score["test_mse"].mean())
        print("r2: ", score["test_r2"].mean())
        print()
    print()

Cu Concentration
mae:  9.622499999999999
mse:  163.748349375
r2:  0.8129773339155146

Zn Concentration
mae:  52.627750000000006
mse:  14010.46368625
r2:  0.8729745988180241

As Concentration
mae:  6.776375
mse:  129.91407125000003
r2:  0.39832765002674625

Pb Concentration
mae:  39.711875
mse:  9604.997845625
r2:  0.9326983296120476


Cu Concentration
mae:  10.591350457734958
mse:  178.73507663663025
r2:  0.7767080167791739

Zn Concentration
mae:  33.01767167205442
mse:  2346.986688912827
r2:  0.9644394681829294

As Concentration
mae:  4.683323465655776
mse:  51.19093044362343
r2:  0.6797359775972327

Pb Concentration
mae:  37.47778593683357
mse:  3288.076524964488
r2:  0.9662662225242136


Cu Concentration
mae:  10.603886426059429
mse:  178.83874737778717
r2:  0.776574489950294

Zn Concentration
mae:  33.02839116083881
mse:  2348.267438291203
r2:  0.9644134680107042

As Concentration
mae:  4.655909618014183
mse:  50.48107302768642
r2:  0.6849633791815426

Pb Concentration
mae:  37.486